In [4]:
# 10_results_aggregation.ipynb - Full ready-to-paste cell
# Aggregates experiments & metrics across pipeline (train/distill/calib/explain/rule/deploy)
# Saves summary CSV/JSON/Excel and plots. Defensive: tolerant to missing files.

import json, math, time, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, brier_score_loss
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
DISTILL_ROOT = ROOT / "outputs" / "distillation"
CALIB_ROOT = ROOT / "outputs" / "calibration"
EXPLAIN_ROOT = ROOT / "outputs" / "explainability"
RULE_ROOT = ROOT / "outputs" / "rule_mapping"
DEPLOY_ROOT = ROOT / "outputs" / "deploy_quantize"
ADV_ROOT = ROOT / "outputs" / "advanced_experiments"
AGG_OUT = ROOT / "outputs" / "results_aggregation"
STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"

ensure_dir = lambda p: p.mkdir(parents=True, exist_ok=True) or p
ensure_dir(AGG_OUT)

# helper functions
def safe_load_json(p):
    try:
        return json.load(open(p, encoding="utf8"))
    except Exception:
        return None
def safe_read_csv(p):
    try:
        return pd.read_csv(p)
    except Exception:
        return None
def human_bytes(n):
    if n is None or n==0: return "0B"
    for unit in ['B','KB','MB','GB','TB']:
        if n < 1024.0:
            return f"{n:3.2f}{unit}"
        n /= 1024.0
    return f"{n:.2f}PB"

def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins=15):
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    accuracies = (preds == labels).astype(float)
    bins = np.linspace(0.0,1.0,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences>bins[i]) & (confidences<=bins[i+1])
        if mask.sum()==0: continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum()/len(probs)) * abs(bin_conf - bin_acc)
    return float(ece)

# ---------- discover experiments ----------
# We expect experiments under outputs/distillation/<lang>/<model_id>/ with predictions.csv and metrics
langs = sorted([d.name for d in (ROOT/"data_splits").iterdir() if d.is_dir()])
experiments = []
for lang in langs:
    exp_dir = DISTILL_ROOT / lang / STUDENT_MODEL_ID
    if not exp_dir.exists():
        # skip if no distillation outputs
        continue
    # predictions: try multiple candidate filenames
    cand_preds = [
        exp_dir / "predictions.csv",
        exp_dir / "predictions_test.csv",
        exp_dir / "student_test_file_predictions_topk_k3.csv",
        exp_dir / "student_test_file_predictions_topk_k3_calibrated.csv",
        exp_dir / "student_test_file_predictions.csv"
    ]
    pred_file = None
    for c in cand_preds:
        if c.exists():
            pred_file = c; break
    if pred_file is None:
        # fallback: any csv in folder with 'pred' in name
        for f in exp_dir.glob("*.csv"):
            if 'pred' in f.name.lower():
                pred_file = f; break
    # metrics file (from distillation eval)
    metrics_candidates = list(exp_dir.glob("*.json")) + list(exp_dir.glob("metrics*.json")) + list(exp_dir.glob("*/metrics.json"))
    metrics_file = None
    for m in metrics_candidates:
        if m.exists() and 'metrics' in m.name.lower():
            metrics_file = m; break
    # calibration artifact
    calib_path = CALIB_ROOT / lang / STUDENT_MODEL_ID / "calibration_summary.json"
    # explainability
    explain_csv = EXPLAIN_ROOT / lang / STUDENT_MODEL_ID / "explainability_metrics_per_file.csv"
    top_tokens_csv = EXPLAIN_ROOT / lang / STUDENT_MODEL_ID / "top_tokens_per_file.csv"
    # rule mapping
    rule_dir = RULE_ROOT / lang / STUDENT_MODEL_ID
    rule_preds = rule_dir / "rule_mapping_predictions.csv"
    rule_summary = rule_dir / "rule_mapping_summary.json"
    # deploy metrics
    deploy_metrics = DEPLOY_ROOT / lang / STUDENT_MODEL_ID / "deploy_metrics.json"
    answer = {
        "lang": lang,
        "distill_dir": str(exp_dir),
        "predictions": str(pred_file) if pred_file else None,
        "distill_metrics": str(metrics_file) if metrics_file else None,
        "calibration_summary": str(calib_path) if calib_path.exists() else None,
        "explainability_metrics": str(explain_csv) if explain_csv.exists() else None,
        "top_tokens": str(top_tokens_csv) if top_tokens_csv.exists() else None,
        "rule_predictions": str(rule_preds) if rule_preds.exists() else None,
        "rule_summary": str(rule_summary) if rule_summary.exists() else None,
        "deploy_metrics": str(deploy_metrics) if deploy_metrics.exists() else None
    }
    experiments.append(answer)

# Save experiments index
json.dump(experiments, open(AGG_OUT/"experiments_index.json","w",encoding="utf8"), indent=2)
print(f"Discovered {len(experiments)} experiments. Index saved to {AGG_OUT/'experiments_index.json'}")

# ---------- aggregate per-experiment ----------
rows = []
per_file_dfs = []
for exp in experiments:
    lang = exp['lang']
    row = {"language": lang}
    # load distillation metrics
    dist_metrics = safe_load_json(exp['distill_metrics']) if exp['distill_metrics'] else None
    if dist_metrics:
        row.update({"distill_metrics_file": exp['distill_metrics']})
        # try to extract key numbers if present
        # many variants: 'val_macro_f1', 'best_val_macro_f1', 'metrics' etc
        for k in ['val_macro_f1','val_f1','best_val_macro_f1','best_val_f1','macro_f1']:
            if k in dist_metrics:
                row['distill_'+k]=dist_metrics[k]
    # load predictions
    pred_df = safe_read_csv(exp['predictions']) if exp['predictions'] else None
    if pred_df is None:
        row['note']="predictions_missing"
        rows.append(row)
        continue
    # normalize predictions (attempt)
    # ensure columns: file_path, pred_label, pred_label_id, pred_probs (list or columns)
    if 'file_path' not in pred_df.columns:
        # try to find a path-like column
        for c in pred_df.columns:
            if 'file' in c.lower() or 'path' in c.lower():
                pred_df = pred_df.rename(columns={c:'file_path'}); break
    # attempt to assemble probs array
    probs_col = None
    if 'pred_probs' in pred_df.columns:
        probs_col = 'pred_probs'
    elif 'pred_probs_norm' in pred_df.columns:
        probs_col = 'pred_probs_norm'
    else:
        # find columns like prob_0, prob_1 ...
        pcols = [c for c in pred_df.columns if c.lower().startswith('prob') or c.lower().startswith('p_') or c.lower().startswith('proba')]
        if pcols:
            probs_col = pcols
    # add final normalized prob array column
    probs_list = []
    pred_label_ids = []
    pred_label_strs = []
    for idx,r in pred_df.iterrows():
        # probs
        probs = None
        if probs_col:
            if isinstance(probs_col, list):
                try:
                    arr=[float(r[c]) if not pd.isnull(r[c]) else 0.0 for c in probs_col]
                    probs = np.array(arr)
                except Exception:
                    probs=None
            else:
                v = r.get(probs_col)
                if isinstance(v, str):
                    try: probs = np.array(json.loads(v))
                    except Exception:
                        try:
                            # maybe string like "[0.1, 0.9]"
                            v2 = v.strip()
                            if v2.startswith('[') and v2.endswith(']'):
                                probs = np.array([float(x.strip()) for x in v2[1:-1].split(',')])
                        except Exception:
                            probs=None
                elif isinstance(v,(list,tuple,np.ndarray)):
                    probs = np.array(v)
        if probs is None:
            # fallback: if pred_label_id present, make one-hot
            pli = None
            if 'pred_label_id' in pred_df.columns and not pd.isnull(r.get('pred_label_id')):
                try: pli=int(r.get('pred_label_id'))
                except: pli=None
            if pli is not None:
                # default size guess:  max label id +1
                num_labels = int(pred_df['pred_label_id'].max()+1) if 'pred_label_id' in pred_df.columns else 2
                p = np.zeros((num_labels,),dtype=float)
                if 0 <= pli < num_labels: p[pli]=1.0
                probs = p
            else:
                # uniform small vector
                probs = np.ones((2,),dtype=float)/2.0
        probs_list.append(probs.tolist())
        # pred label id
        if 'pred_label_id' in pred_df.columns and not pd.isnull(r.get('pred_label_id')):
            try: pred_label_ids.append(int(r.get('pred_label_id')))
            except Exception: pred_label_ids.append(int(np.argmax(probs)))
        else:
            pred_label_ids.append(int(np.argmax(probs)))
        if 'pred_label' in pred_df.columns and not pd.isnull(r.get('pred_label')):
            pred_label_strs.append(str(r.get('pred_label')))
        else:
            pred_label_strs.append(str(pred_label_ids[-1]))
    pred_df['pred_probs_array'] = probs_list
    pred_df['pred_label_id_norm'] = pred_label_ids
    pred_df['pred_label_norm'] = pred_label_strs

    # attempt to compute per-experiment metrics vs gold if gold exists in pred_df or test split
    # gold may be in pred_df as 'label' or 'gold_label' or else load test split
    if 'label' in pred_df.columns and pred_df['label'].notnull().sum()>0:
        gold_col = 'label'
    elif 'gold_label' in pred_df.columns and pred_df['gold_label'].notnull().sum()>0:
        gold_col = 'gold_label'
    else:
        # try load test split from data_splits
        test_csv = ROOT / "data_splits" / lang / "test.csv"
        test_df = safe_read_csv(test_csv) if test_csv.exists() else None
        if test_df is not None and 'file_path' in test_df.columns and 'label' in test_df.columns:
            # merge
            merged = pd.merge(test_df[['file_path','label']], pred_df, on='file_path', how='left')
            pred_df = merged
            gold_col = 'label'
        else:
            gold_col = None

    # compute evaluation if gold available
    if gold_col is not None and pred_df[gold_col].notnull().sum()>0:
        # we need a label_map to turn label strings to ints; try to read label_map.json
        label_map_path = ROOT / "data_splits" / lang / "label_map.json"
        label_map = {}
        inv_label_map = {}
        if label_map_path.exists():
            jm = safe_load_json(label_map_path)
            label_map = {str(k):int(v) for k,v in jm.get("label_map",{}).items()}
            inv_label_map = {int(v):str(k) for k,v in jm.get("label_map",{}).items()}
        # map gold strings to ids (if necessary)
        gold_ids = []
        for _,r in pred_df.iterrows():
            g = r.get(gold_col)
            if pd.isnull(g):
                gold_ids.append(None)
            else:
                if g in label_map:
                    gold_ids.append(int(label_map[g]))
                else:
                    # maybe stored as id
                    try:
                        gold_ids.append(int(g))
                    except Exception:
                        gold_ids.append(None)
        pred_df['gold_id'] = gold_ids
        # compute metrics on rows where gold_id exists
        valid = pred_df['gold_id'].notnull()
        y_true = np.array(pred_df.loc[valid,'gold_id'].astype(int).tolist())
        y_pred = np.array(pred_df.loc[valid,'pred_label_id_norm'].astype(int).tolist())
        # Fix for ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
        probs_stack = np.vstack(pred_df.loc[valid,'pred_probs_array'].apply(lambda x: np.array(x) if not pd.isna(x).all() else np.ones((probs_stack.shape[1],))/probs_stack.shape[1] if 'probs_stack' in locals() else np.ones((2,))/2.0).tolist())

        acc = accuracy_score(y_true, y_pred)
        p,r,f,_ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        cm = confusion_matrix(y_true, y_pred).tolist()
        # ECE & Brier
        try:
            ece = expected_calibration_error(probs_stack, y_true, n_bins=15)
        except Exception:
            ece = None
        try:
            brier = float(np.mean([brier_score_loss((y_true==k).astype(int), probs_stack[:,k]) for k in range(probs_stack.shape[1])]))
        except Exception:
            brier = None
        row.update({
            "n_total_files": int(len(pred_df)),
            "n_with_gold": int(valid.sum()),
            "accuracy": float(acc),
            "macro_f1": float(f),
            "ece_top_label": ece,
            "brier_mean": brier,
            "confusion_matrix": cm
        })
    else:
        row.update({"note":"no_gold_available", "n_total_files": int(len(pred_df))})

    # attach explainability aggregated metrics if available
    if exp['explainability_metrics']:
        df_expl = safe_read_csv(exp['explainability_metrics'])
        if df_expl is not None and 'file_path' in df_expl.columns:
            # compute mean deletion_auc, insertion_auc etc if present
            for col in ['deletion_auc','insertion_auc','attr_entropy','stability_cosine','sparsity_frac_for_90pct']:
                if col in df_expl.columns:
                    try:
                        row['expl_'+col+'_mean'] = float(df_expl[col].dropna().astype(float).mean())
                    except Exception:
                        row['expl_'+col+'_mean'] = None
    # attach rule mapping summary if available
    if exp['rule_summary']:
        js = safe_load_json(exp['rule_summary'])
        if js:
            # final_metrics likely present
            fm = js.get("final_metrics", None)
            if fm and 'accuracy' in fm:
                row['rule_final_accuracy'] = fm['accuracy']
            # save path
            row['rule_summary_file'] = exp['rule_summary']

    # attach deploy metrics size if available
    if exp['deploy_metrics']:
        dm = safe_load_json(exp['deploy_metrics'])
        if dm:
            # try to extract quantized size and latency
            q = dm.get('outputs',{}).get('quantized_dynamic',{})
            if q:
                row['quantized_dir_bytes'] = q.get('size_bytes') or q.get('size_bytes', None)
            row['latency_ms_per_sample'] = dm.get('latency_ms_per_sample')
            # original model size may be present in deploy_metrics json under 'orig_model_size_bytes'
            row['orig_model_size_bytes'] = dm.get('orig_model_size_bytes')
    rows.append(row)
    # save per-file for downstream analyses
    pred_df['language'] = lang
    per_file_dfs.append(pred_df)

# ---------- save aggregated tables ----------
df_summary = pd.DataFrame(rows)
df_summary.to_csv(AGG_OUT/"experiments_summary.csv", index=False)
json.dump(rows, open(AGG_OUT/"experiments_summary.json","w",encoding="utf8"), indent=2)
print("Saved experiments_summary.csv/json to", AGG_OUT)

# Save combined per-file table
if per_file_dfs:
    df_perfile = pd.concat(per_file_dfs, ignore_index=True, sort=False)
    df_perfile.to_csv(AGG_OUT/"predictions_per_file_combined.csv", index=False)
    print("Saved per-file combined predictions to", AGG_OUT/"predictions_per_file_combined.csv")
else:
    df_perfile = pd.DataFrame()

# ---------- produce plots ----------
# 1) bar: accuracy and macro-f1 per language
plt.figure(figsize=(8,4))
plot_df = df_summary.copy()
plot_df['accuracy_plot'] = plot_df['accuracy'].fillna(0)
plot_df = plot_df.sort_values('accuracy_plot', ascending=False)
plt.bar(plot_df['language'], plot_df['accuracy_plot'], label='accuracy')
plt.ylabel("Accuracy")
plt.title("Per-language accuracy")
plt.savefig(AGG_OUT/"accuracy_per_language.png", dpi=150)
plt.close()

# 2) ECE plot if available
if 'ece_top_label' in df_summary.columns and df_summary['ece_top_label'].notnull().sum()>0:
    plt.figure(figsize=(8,4))
    ece_df = df_summary[['language','ece_top_label']].dropna().sort_values('ece_top_label')
    plt.bar(ece_df['language'], ece_df['ece_top_label'])
    plt.ylabel("ECE (top-label)")
    plt.title("Calibration (ECE) per language")
    plt.savefig(AGG_OUT/"ece_per_language.png", dpi=150)
    plt.close()

# 3) model size bar (orig and quantized if present)
size_cols = []
if 'orig_model_size_bytes' in df_summary.columns and df_summary['orig_model_size_bytes'].notnull().sum()>0:
    size_cols.append('orig_model_size_bytes')
if 'quantized_dir_bytes' in df_summary.columns and df_summary['quantized_dir_bytes'].notnull().sum()>0:
    size_cols.append('quantized_dir_bytes')
if size_cols:
    plt.figure(figsize=(8,4))
    for col in size_cols:
        df_summary[col+'_mb'] = df_summary[col].apply(lambda x: (int(x)/(1024*1024)) if (not pd.isnull(x) and x is not None) else np.nan)
    x = np.arange(len(df_summary))
    width = 0.35
    plt.bar(df_summary['language'], df_summary[size_cols[0]+'_mb'], width, label=size_cols[0])
    if len(size_cols)>1:
        plt.bar(df_summary['language'], df_summary[size_cols[1]+'_mb'], width, bottom=df_summary[size_cols[0]+'_mb'], label=size_cols[1])
    plt.ylabel("Model size (MB)")
    plt.title("Model sizes")
    plt.legend()
    plt.savefig(AGG_OUT/"model_sizes.png", dpi=150)
    plt.close()

# 4) Summary Excel for paper
try:
    excel_path = AGG_OUT/"results_summary.xlsx"
    with pd.ExcelWriter(excel_path) as writer:
        df_summary.to_excel(writer, sheet_name="experiments_summary", index=False)
        if not df_perfile.empty:
            df_perfile.to_excel(writer, sheet_name="predictions_per_file", index=False)
    print("Saved Excel summary to", excel_path)
except Exception as e:
    print("Excel write failed:", e)

# ---------- print a short report ----------
print("\n========= Aggregation Report =========")
print("Experiments found:", len(experiments))
print("Summary table saved to:", AGG_OUT/"experiments_summary.csv")
if not df_perfile.empty:
    print("Per-file combined saved to:", AGG_OUT/"predictions_per_file_combined.csv")
if (AGG_OUT/"results_summary.xlsx").exists():
    print("Excel workbook:", AGG_OUT/"results_summary.xlsx")
print("Figures saved under:", AGG_OUT)
print("======================================\n")

# ---------- additional tip for paper: LaTeX table output ----------
# Convert key columns into a LaTeX-friendly table (accuracy, macro_f1, ece, sizes)
latex_df = df_summary[['language','accuracy','macro_f1','ece_top_label','orig_model_size_bytes','quantized_dir_bytes']].copy()
latex_df.to_csv(AGG_OUT/"latex_table_ready.csv", index=False)
try:
    with open(AGG_OUT/"latex_table.tex","w",encoding="utf8") as f:
        f.write(latex_df.to_latex(index=False, na_rep="NA", float_format="%.3f"))
    print("Saved LaTeX table to", AGG_OUT/"latex_table.tex")
except Exception:
    pass

print("Aggregation finished. If something is missing, inspect experiments_index.json and individual experiment folders listed there.")

Discovered 3 experiments. Index saved to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/experiments_index.json
Saved experiments_summary.csv/json to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation
Saved per-file combined predictions to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/predictions_per_file_combined.csv
Saved Excel summary to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/results_summary.xlsx

========= Aggregation Report =========
Experiments found: 3
Summary table saved to: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/experiments_summary.csv
Per-file combined saved to: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/predictions_per_file_combined.csv
Exce

In [5]:
# Extra analysis: per-class heatmaps, paired-bootstrap significance, and multi-panel figures
# Paste into 10_results_aggregation.ipynb below previous aggregation cell (or run in a fresh cell).
import json, math, time, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score
import itertools
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
AGG_OUT = ROOT / "outputs" / "results_aggregation"
EXTRA_OUT = AGG_OUT / "extra_analysis"
ensure_dir = lambda p: p.mkdir(parents=True, exist_ok=True) or p
ensure_dir(EXTRA_OUT)

# Files produced by previous aggregation
perfile_csv = AGG_OUT / "predictions_per_file_combined.csv"
experiments_index_path = AGG_OUT / "experiments_index.json"

if not perfile_csv.exists():
    raise FileNotFoundError(f"Required file not found: {perfile_csv} — run aggregation step first.")

# Load per-file predictions (combined)
df = pd.read_csv(perfile_csv, dtype=object)  # keep flexible types
# Normalize some columns if present
# expected columns created earlier: file_path, language, gold_id, pred_label_id (model), maybe final_label_id, rule_label etc.
# Try to detect columns for different systems
cols = df.columns.tolist()

# Normalize numeric columns that may be strings
def to_int_or_none(x):
    try:
        if pd.isna(x): return None
        return int(float(x))
    except Exception:
        return None

# Column detection heuristics
gold_col = None
if 'gold_id' in df.columns:
    gold_col = 'gold_id'
elif 'label_id' in df.columns:
    gold_col = 'label_id'
else:
    # try to guess columns containing 'gold' or 'label' in name
    for c in df.columns:
        if 'gold' in c.lower() and 'id' in c.lower(): gold_col = c; break

model_col = None
for c in df.columns:
    if c.lower().startswith('pred_label_id') and 'final' not in c.lower() and 'rule' not in c.lower():
        model_col = c; break
if model_col is None:
    # fallback common name
    for c in df.columns:
        if c.lower() in ('pred_label_id_norm','pred_label_id','pred_label'):
            model_col = c; break

final_col = None
# rule mapping final label column may be 'final_label_id' or 'final_label_id' in rule_mapping outputs merged earlier
for c in df.columns:
    if 'final_label' in c.lower() and 'id' in c.lower():
        final_col = c; break
# also try 'final_label_id' or 'final_label'
if final_col is None and 'final_label_id' in df.columns:
    final_col = 'final_label_id'

rule_col_candidate = None
for c in df.columns:
    if 'rule' in c.lower() and 'label' in c.lower() and 'id' in c.lower():
        rule_col_candidate = c; break

# Some pipelines may have produced 'final_label_id' in rule mapping. Capture possible names
possible_system_cols = {
    "model": model_col,
    "final": final_col,
    "rule": rule_col_candidate
}
print("Detected columns (may be None):", possible_system_cols, "gold_col =", gold_col)

# cast to ints where possible
if gold_col:
    df['_gold_id'] = df[gold_col].apply(to_int_or_none)
else:
    df['_gold_id'] = None

for sysname, col in possible_system_cols.items():
    if col and col in df.columns:
        df[f'__{sysname}_id'] = df[col].apply(to_int_or_none)
    else:
        df[f'__{sysname}_id'] = None

# Helper: compute per-class precision/recall/f1 for a given (y_true, y_pred)
def per_class_f1(y_true, y_pred, labels=None):
    # labels: list of ints to include; if none, infer from union
    if len(y_true)==0:
        return {}
    if labels is None:
        labels = sorted(list(set([int(x) for x in y_true if x is not None] + [int(x) for x in y_pred if x is not None])))
    p,r,f,_ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {labels[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i])} for i in range(len(labels))}

# Build per-language and per-system per-class F1 table
languages = sorted(df['language'].dropna().unique().tolist())
systems = ['model','rule','final']
per_class_table = {}  # structure: per_class_table[lang][system][class_id] = f1
for lang in languages:
    per_class_table[lang] = {}
    sub = df[df['language']==lang].copy()
    # select only rows with gold
    sub_valid = sub[sub['_gold_id'].notnull()].copy()
    if sub_valid.empty:
        print(f"[WARN] No gold labels for language {lang}; skipping per-class for this language.")
        continue
    y_true = [int(x) for x in sub_valid['_gold_id'].tolist()]
    # candidate label set
    labels_set = sorted(list({int(x) for x in y_true if x is not None}))
    # for each system, compute per-class f1
    for sysname in systems:
        col = f'__{sysname}_id'
        if col in sub_valid.columns and sub_valid[col].notnull().sum()>0:
            y_pred = [int(x) if x is not None else -1 for x in sub_valid[col].tolist()]
            # ensure length matches
            # compute per-class f1 for labels_set
            pf = per_class_f1(y_true, y_pred, labels=labels_set)
            per_class_table[lang][sysname] = pf
        else:
            per_class_table[lang][sysname] = {}

# Create heatmaps: rows=class labels, cols=(lang,system) multiindex
# Build a wide DataFrame where index is class label (string) and columns are MultiIndex (lang, system)
heat_dfs = {}
for lang in languages:
    # first collect all classes present across systems for this lang
    class_ids = sorted({int(k) for sys in per_class_table[lang].keys() for k in per_class_table[lang][sys].keys()})
    if not class_ids:
        continue
    df_heat = pd.DataFrame(index=[str(c) for c in class_ids])
    for sysname in systems:
        vals = []
        for cid in class_ids:
            f1 = per_class_table[lang].get(sysname, {}).get(cid, {}).get('f1', np.nan)
            vals.append(f1)
        df_heat[sysname] = vals
    heat_dfs[lang] = df_heat
    # save heatmap for this language
    plt.figure(figsize=(6, max(3, len(class_ids)*0.4)))
    sns.heatmap(df_heat.astype(float), annot=True, fmt=".3f", cmap="viridis", cbar_kws={'label':'F1'})
    plt.title(f"Per-class F1 across systems — {lang}")
    plt.ylabel("class id"); plt.xlabel("system")
    plt.tight_layout()
    ppath = EXTRA_OUT / f"per_class_heatmap_{lang}.png"
    plt.savefig(ppath, dpi=150)
    plt.close()
    print(f"[SAVED] per-class heatmap for {lang} -> {ppath}")

# ---------------- Paired bootstrap test for macro-F1 differences ----------------
# This compares two systems on paired files (same file-level instances). We'll test:
#   model vs final (if both available); otherwise, model vs rule if available.
def macro_f1_on_pairs(y_true, y_pred):
    # use sklearn's macro f1 via precision_recall_fscore_support
    _,_,f,_ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return float(f)

def paired_bootstrap_macroF1(y_true, pred_a, pred_b, n_resamples=5000, seed=42):
    """
    Returns dict with observed_diff = macroF1(a)-macroF1(b), p_value (two-sided),
    and 95% bootstrap CI for the difference.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    a_vals = np.array(pred_a)
    b_vals = np.array(pred_b)
    base_a = macro_f1_on_pairs(y_true, a_vals)
    base_b = macro_f1_on_pairs(y_true, b_vals)
    obs_diff = base_a - base_b
    diffs = []
    for i in range(n_resamples):
        idxs = rng.randint(0, n, size=n)  # sample with replacement paired indices
        ya = np.array(y_true)[idxs]
        aa = a_vals[idxs]
        bb = b_vals[idxs]
        fa = macro_f1_on_pairs(ya, aa)
        fb = macro_f1_on_pairs(ya, bb)
        diffs.append(fa - fb)
    diffs = np.array(diffs)
    # two-sided p-value: proportion of bootstrap diffs whose sign is opposite to obs_diff (simple approx)
    if obs_diff >= 0:
        p_value = np.mean(diffs <= 0)
    else:
        p_value = np.mean(diffs >= 0)
    ci_low = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)
    return {"obs_diff": obs_diff, "p_value": float(p_value), "ci_2.5": float(ci_low), "ci_97.5": float(ci_high), "bootstrap_diffs_sample": diffs[:100].tolist()}

# run paired tests per-language
paired_results = {}
for lang in languages:
    sub = df[df['language']==lang].copy()
    valid = sub[sub['_gold_id'].notnull()].copy()
    if valid.empty:
        continue
    y_true = [int(x) for x in valid['_gold_id'].tolist()]
    # choose systems to compare
    # prefer model vs final, else model vs rule
    a_col = '__model_id'
    b_col = None
    if '__final_id' in valid.columns and valid['__final_id'].notnull().sum()>0:
        b_col = '__final_id'
    elif '__rule_id' in valid.columns and valid['__rule_id'].notnull().sum()>0:
        b_col = '__rule_id'
    else:
        # nothing to compare
        continue
    a_preds = [to_int_or_none(x) for x in valid[a_col].tolist()]
    b_preds = [to_int_or_none(x) for x in valid[b_col].tolist()]
    # replace None with model prediction fallback if needed
    a_preds = [int(x) if x is not None else 0 for x in a_preds]
    b_preds = [int(x) if x is not None else 0 for x in b_preds]
    res = paired_bootstrap_macroF1(y_true, a_preds, b_preds, n_resamples=2000, seed=2025)
    paired_results[lang] = {"system_A":"model", "system_B": "final" if b_col=='__final_id' else "rule", "paired_test": res}
    # save per-lang result
    json.dump(res, open(EXTRA_OUT / f"paired_bootstrap_{lang}.json","w",encoding="utf8"), indent=2)
    print(f"[PAIRED] {lang}: obs_diff={res['obs_diff']:.4f}, p={res['p_value']:.4f}, CI=({res['ci_2.5']:.4f},{res['ci_97.5']:.4f})")

# save paired summary
json.dump(paired_results, open(EXTRA_OUT/"paired_bootstrap_summary.json","w",encoding="utf8"), indent=2)

# ---------------- Multi-panel figures per language ----------------
# For each language: reliability diagram (top-label) and confusion matrices for model vs final side-by-side.
from math import ceil
def plot_reliability(probs_arr, labels_arr, outpath, n_bins=15, title=None):
    confidences = probs_arr.max(axis=1)
    preds = probs_arr.argmax(axis=1)
    accuracies = (preds==labels_arr).astype(float)
    bins = np.linspace(0,1,n_bins+1)
    bin_centers=[]
    bin_accs=[]
    bin_confs=[]
    for i in range(n_bins):
        mask = (confidences>bins[i]) & (confidences<=bins[i+1])
        if mask.sum()==0:
            bin_centers.append((bins[i]+bins[i+1])/2)
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)
        else:
            bin_centers.append((bins[i]+bins[i+1])/2)
            bin_accs.append(accuracies[mask].mean())
            bin_confs.append(confidences[mask].mean())
    plt.figure(figsize=(5,5))
    plt.plot([0,1],[0,1],'k:', label='perfect')
    plt.plot(bin_centers, [0 if math.isnan(x) else x for x in bin_accs], marker='o', label='accuracy')
    plt.xlabel("Confidence"); plt.ylabel("Accuracy")
    if title: plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

for lang in languages:
    sub = df[df['language']==lang].copy()
    valid = sub[sub['_gold_id'].notnull()].copy()
    if valid.empty:
        continue
    y_true = np.array([int(x) for x in valid['_gold_id'].tolist()])
    # find per-file probs array if present (column name may vary: 'pred_probs' or 'pred_probs_array' or 'pred_probs_norm')
    probs_col_candidates = [c for c in valid.columns if 'pred_probs' in c]
    probs_arr = None
    if probs_col_candidates:
        # take first match
        pc = probs_col_candidates[0]
        try:
            probs_arr = np.vstack(valid[pc].apply(lambda x: np.array(json.loads(x)) if isinstance(x,str) and x.strip().startswith('[') else (np.array(x) if isinstance(x,(list,tuple,np.ndarray)) else np.array([float(x)]))).tolist())
        except Exception:
            # fallback: try eval per element
            arrs=[]
            for v in valid[pc].tolist():
                try:
                    if isinstance(v, str) and v.strip().startswith('['):
                        arrs.append(np.array(json.loads(v)))
                    elif isinstance(v,(list,tuple,np.ndarray)):
                        arrs.append(np.array(v))
                    else:
                        arrs.append(np.array([float(v)]))
                except Exception:
                    arrs.append(np.array([0.5,0.5]))
            # pad to max length if needed
            maxlen = max([a.shape[0] for a in arrs])
            arrp = np.array([np.pad(a, (0,maxlen-a.shape[0]), constant_values=0.0) for a in arrs])
            probs_arr = arrp
    # if no probs arr, skip reliability
    lang_out = EXTRA_OUT / lang
    ensure_dir(lang_out)
    if probs_arr is not None and probs_arr.shape[0]==len(y_true):
        try:
            plot_reliability(probs_arr, y_true, lang_out / "reliability_toplabel.png", n_bins=15, title=f"{lang} reliability (top-label)")
            print(f"[SAVED] reliability diagram for {lang} -> {lang_out/'reliability_toplabel.png'}")
        except Exception as e:
            print("[WARN] reliability plotting failed for", lang, e)
    # Confusion matrices for model and final (if available)
    model_preds = valid['__model_id'].apply(lambda x: to_int_or_none(x)).tolist()
    final_preds = None
    if '__final_id' in valid.columns and valid['__final_id'].notnull().sum()>0:
        final_preds = valid['__final_id'].apply(lambda x: to_int_or_none(x)).tolist()
    # derive label set
    label_ids = sorted(list(set([int(x) for x in y_true.tolist() if x is not None] + [int(x) for x in model_preds if x is not None])))
    if len(label_ids)==0:
        continue
    # confusion matrix plot
    cm_model = confusion_matrix(y_true, [int(x) if x is not None else -1 for x in model_preds], labels=label_ids)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.heatmap(cm_model, annot=True, fmt="d", cmap="Blues", xticklabels=[str(i) for i in label_ids], yticklabels=[str(i) for i in label_ids])
    plt.xlabel("Pred"); plt.ylabel("True"); plt.title(f"{lang} Confusion (model)")
    if final_preds is not None:
        cm_final = confusion_matrix(y_true, [int(x) if x is not None else -1 for x in final_preds], labels=label_ids)
        plt.subplot(1,2,2)
        sns.heatmap(cm_final, annot=True, fmt="d", cmap="Blues", xticklabels=[str(i) for i in label_ids], yticklabels=[str(i) for i in label_ids])
        plt.xlabel("Pred"); plt.ylabel("True"); plt.title(f"{lang} Confusion (final)")
    plt.tight_layout()
    fpath = lang_out / "confusion_model_vs_final.png"
    plt.savefig(fpath, dpi=150)
    plt.close()
    print(f"[SAVED] confusion matrices for {lang} -> {fpath}")

# ---------------- Save combined per-class table (CSV) ----------------
combined_rows=[]
for lang, d in per_class_table.items():
    for sysname, mapping in d.items():
        for cid, metrics in mapping.items():
            combined_rows.append({"language": lang, "system": sysname, "class_id": cid, "f1": metrics.get('f1'), "precision": metrics.get('precision'), "recall": metrics.get('recall')})
df_combined = pd.DataFrame(combined_rows)
if not df_combined.empty:
    df_combined.to_csv(EXTRA_OUT/"per_class_f1_table.csv", index=False)
    print("[SAVED] per_class_f1_table.csv ->", EXTRA_OUT/"per_class_f1_table.csv")

# ---------------- final notes & outputs ----------------
print("\n[EXTRA_ANALYSIS DONE]")
print("Outputs saved under:", EXTRA_OUT)
print("- per-class heatmaps: per_class_heatmap_<lang>.png")
print("- paired bootstrap JSONs: paired_bootstrap_<lang>.json")
print("- per-class CSV table: per_class_f1_table.csv")
print("- multi-panel per-language figs: <lang>/reliability_toplabel.png and <lang>/confusion_model_vs_final.png")


Detected columns (may be None): {'model': 'pred_label_id', 'final': None, 'rule': None} gold_col = gold_id
[SAVED] per-class heatmap for English -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/extra_analysis/per_class_heatmap_English.png
[SAVED] per-class heatmap for Hindi -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/extra_analysis/per_class_heatmap_Hindi.png
[SAVED] per-class heatmap for Marathi -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/extra_analysis/per_class_heatmap_Marathi.png
[SAVED] reliability diagram for English -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/extra_analysis/English/reliability_toplabel.png
[SAVED] confusion matrices for English -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/results_aggregation/e